In [2]:
import sys
import os
import json
import urllib3
import requests
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)


Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [ ]:
import pandas as pd
from datetime import datetime

# Define time period
start = "2025-01-01T00:00:00Z"  # adjust as needed

# List of owners to pull from
list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        # Build TQL query string
        tql = f'ownerName EQ "{owner}" and lastObserved >= "{start}" and (indicatorActive EQ true OR indicatorActive EQ false)'

        # Set up the request
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql}&fields=threatAssess,associatedGroups,observations&resultStart=0&resultLimit=10000')

        # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    observed_src = pd.json_normalize(final_results, record_path='data')
    observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
    print(f"\nRetrieved {len(observed_src)} observed indicators.")
else:
    print("\nNo indicators retrieved.")



Querying owner: HTOC Org

Querying owner: HTOC Comm

Querying owner: HTOC legacy data

Retrieved 1383 observed indicators.


In [4]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,ip,legacyLink,associatedGroups.data,hostName,dnsActive,whoisActive,address,url,text,indicator
0,5288144,2025-02-04T19:30:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-24T15:10:00Z,4.0,89.0,4.0,...,65.21.203.39,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 5629499536006730, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,65.21.203.39
1,4557470,2024-04-04T14:26:25Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-24T14:45:26Z,4.0,60.0,2.0,...,80.66.83.49,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 329054, 'dateAdded': '2024-04-08T14:49...",NaN,NaN,NaN,NaN,NaN,NaN,80.66.83.49
2,4839539,2024-08-21T18:02:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-24T14:45:11Z,3.0,27.0,1.5,...,185.224.128.59,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 451934, 'dateAdded': '2024-08-21T18:02...",NaN,NaN,NaN,NaN,NaN,NaN,185.224.128.59
3,5272639,2025-01-31T11:46:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T14:42:08Z,2.0,21.0,1.0,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 548878, 'dateAdded': '2025-01-31T11:46...",sparkcarwash.com,False,False,NaN,NaN,NaN,sparkcarwash.com
4,3975798,2021-10-14T15:11:42Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-24T14:42:02Z,3.0,0.0,3.5,...,154.216.69.38,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 126366, 'dateAdded': '2021-10-14T14:06...",NaN,NaN,NaN,NaN,NaN,NaN,154.216.69.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1378,4482005,2023-11-30T16:50:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2023-12-01T11:40:35Z,3.0,88.0,3.0,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 289535, 'dateAdded': '2023-11-30T16:47...",NaN,NaN,NaN,NaN,pub.marq.com/,NaN,pub.marq.com/
1379,3671024,2021-03-17T15:07:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,EmailAddress,2023-10-23T16:35:18Z,3.0,91.0,3.0,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 114647, 'dateAdded': '2021-03-17T14:35...",NaN,NaN,NaN,alsumood@emirates.net.ae,NaN,NaN,alsumood@emirates.net.ae
1380,4064075,2022-01-10T15:57:12Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2023-10-11T06:33:38Z,4.0,0.0,2.0,...,3.22.30.40,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 131063, 'dateAdded': '2022-01-20T18:48...",NaN,NaN,NaN,NaN,NaN,NaN,3.22.30.40
1381,4352799,2023-06-08T16:47:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2023-06-09T10:26:16Z,3.0,61.0,3.0,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 153532, 'dateAdded': '2023-06-08T15:36...",NaN,NaN,NaN,NaN,www.emergencylighting.com/,NaN,www.emergencylighting.com/


In [60]:
final_results

[{'data': [], 'status': 'Success'},
 {'data': [], 'status': 'Success'},
 {'data': [{'id': 2469435,
    'dateAdded': '2019-11-06T17:20:14Z',
    'ownerId': 22,
    'ownerName': 'HTOC legacy data',
    'webLink': 'https://hvs.threatconnect.com/#/details/indicators/2469435',
    'type': 'Address',
    'lastModified': '2025-04-22T18:40:03Z',
    'rating': 4.0,
    'confidence': 91,
    'threatAssessRating': 3.58,
    'threatAssessConfidence': 45.92,
    'threatAssessScore': 442,
    'threatAssessScoreObserved': 111,
    'threatAssessScoreFalsePositive': 0,
    'summary': '109.169.86.13',
    'privateFlag': False,
    'active': True,
    'activeLocked': False,
    'ip': '109.169.86.13',
    'legacyLink': 'https://hvs.threatconnect.com/auth/indicators/details/address.xhtml?address=109.169.86.13&owner=HTOC+legacy+data'},
   {'id': 2686837,
    'dateAdded': '2020-07-31T17:28:01Z',
    'ownerId': 22,
    'ownerName': 'HTOC legacy data',
    'webLink': 'https://hvs.threatconnect.com/#/details/in

OPTIONS request failed with status code: 401
Response: {"message":"ERROR (Unauthorized): You are not authorized to perform the requested operation.","status":"Error"}
